# Lung Tumor Segmentation with 3D UNet — NSCLC Radiomics NIfTI
### Fixed: Resize/Pad guarantee | 10 Epochs | Best Model Saving | Loss & Dice Tracking

## 1. Import Required Libraries

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import torch
import warnings
warnings.filterwarnings('ignore')

import pytorch_lightning as pl
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
    Orientationd, ScaleIntensityRanged, CropForegroundd,
    RandCropByPosNegLabeld, SpatialPadd, ResizeWithPadOrCropd,
    RandFlipd, RandRotate90d, RandShiftIntensityd, RandAffined,
    EnsureTyped, AsDiscrete
)
from monai.data import Dataset, decollate_batch
from monai.inferers import sliding_window_inference
from sklearn.model_selection import train_test_split

print("All libraries imported successfully.")


W0421 00:22:00.179000 14292 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


All libraries imported successfully.


## 2. Set Directories and Configuration

In [18]:
# ─── USER CONFIGURATION ───────────────────────────────────────────
IMAGES_DIR = "E:/aryan/nifti_data/images"
LABELS_DIR = "E:/aryan/nifti_data/labels"
OUTPUT_DIR = "E:/aryan/nifti_data/outputs"
PRED_DIR   = "E:/aryan/nifti_data/outputs/predictions"

# ─── QUICK TEST CONFIGURATION (RTX 5090) ──────────────────────────
IMAGES_DIR = "E:/aryan/nifti_data/images"
LABELS_DIR = "E:/aryan/nifti_data/labels"
OUTPUT_DIR = "E:/aryan/nifti_data/outputs"
PRED_DIR   = "E:/aryan/nifti_data/outputs/predictions"

# ─── 10-EPOCH SMOKE TEST ──────────────────────────────────────────
MAX_EPOCHS    = 10
BATCH_SIZE    = 2
ACCUM_GRAD    = 1
SPATIAL_SIZE  = (128, 128, 64)   # smaller than full — faster per epoch
NUM_SAMPLES   = 4
VAL_SPLIT     = 0.2
RANDOM_SEED   = 42
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
POS_NEG_RATIO = 3
# ──────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PRED_DIR,   exist_ok=True)

print(f"Images dir : {IMAGES_DIR}")
print(f"Labels dir : {LABELS_DIR}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Pred dir   : {PRED_DIR}")


Images dir : E:/aryan/nifti_data/images
Labels dir : E:/aryan/nifti_data/labels
Output dir : E:/aryan/nifti_data/outputs
Pred dir   : E:/aryan/nifti_data/outputs/predictions


## 3. Set Device and Reproducibility

In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Selected device: {device}")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

print("Device and seeds configured.")


Selected device: cuda
Device and seeds configured.


## 4. 3D UNet Architecture

In [27]:
class DoubleConv(torch.nn.Module):
    """Two sequential 3-D Conv → BN → ReLU blocks."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = torch.nn.Sequential(
            torch.nn.Conv3d(in_ch,  out_ch, 3, padding=1, bias=False),
            torch.nn.BatchNorm3d(out_ch),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            torch.nn.BatchNorm3d(out_ch),
            torch.nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet3D(torch.nn.Module):
    """3-D UNet: 3 encoder levels + bottleneck + 3 decoder levels."""
    def __init__(self):
        super().__init__()
        self.enc1       = DoubleConv(1,   64)
        self.enc2       = DoubleConv(64,  128)
        self.enc3       = DoubleConv(128, 256)
        self.bottleneck = DoubleConv(256, 512)
        self.dec3       = DoubleConv(512 + 256, 256)
        self.dec2       = DoubleConv(256 + 128, 128)
        self.dec1       = DoubleConv(128 + 64,  64)
        self.out        = torch.nn.Conv3d(64, 1, kernel_size=1)
        self.pool       = torch.nn.MaxPool3d(2)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = torch.nn.functional.interpolate(b,  scale_factor=2, mode='trilinear', align_corners=False)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = torch.nn.functional.interpolate(d3, scale_factor=2, mode='trilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = torch.nn.functional.interpolate(d2, scale_factor=2, mode='trilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out(d1)

print("UNet3D architecture defined.")


UNet3D architecture defined.


## 5. PyTorch Lightning Module

In [36]:
class TumorSegmentation3D(pl.LightningModule):
    """
    Lightning wrapper around UNet3D.
    Logs train_loss, val_loss, val_dice every epoch.
    """
    def __init__(self, lr: float = 1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model        = UNet3D()
        self.loss_fn      = DiceCELoss(sigmoid=True)
        self.dice_metric  = DiceMetric(include_background=False, reduction="mean")
        # History buffers for plotting
        self.train_loss_history  = []
        self.val_loss_history    = []
        self.val_dice_history    = []
        self._train_losses_epoch = []
        self._val_losses_epoch   = []  # buffer for val losses

    def forward(self, x):
        return self.model(x)

    # ── training step ────────────────────────────────────────────────
    def training_step(self, batch, batch_idx):
        # Robust batch handling for dict, tuple/list, or list of dicts
        if isinstance(batch, dict):
            images, labels = batch["image"], batch["label"]
        elif isinstance(batch, (tuple, list)) and len(batch) == 2:
            images, labels = batch
        elif isinstance(batch, list) and len(batch) > 0 and isinstance(batch[0], dict):
            images = torch.cat([b["image"] for b in batch], dim=0)
            labels = torch.cat([b["label"] for b in batch], dim=0)
        else:
            raise TypeError(f"Unexpected batch type: {type(batch)}")
        loss = self.loss_fn(self(images), labels)
        self._train_losses_epoch.append(loss.detach().cpu())
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    # ── validation step ──────────────────────────────────────────────
    def validation_step(self, batch, batch_idx):
        if isinstance(batch, dict):
            images, labels = batch["image"], batch["label"]
        elif isinstance(batch, (tuple, list)) and len(batch) == 2:
            images, labels = batch
        elif isinstance(batch, list) and len(batch) > 0 and isinstance(batch[0], dict):
            images = torch.cat([b["image"] for b in batch], dim=0)
            labels = torch.cat([b["label"] for b in batch], dim=0)
        else:
            raise TypeError(f"Unexpected batch type: {type(batch)}")
        logits = sliding_window_inference(
            inputs=images,
            roi_size=SPATIAL_SIZE,
            sw_batch_size=1,
            predictor=self,
            overlap=0.5,
        )
        val_loss = self.loss_fn(logits, labels)
        self.log("val_loss", val_loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self._val_losses_epoch.append(val_loss.detach().cpu())

        # Robust decollate for both batched and single-tensor cases
        def robust_decollate(tensor):
            try:
                out = decollate_batch(tensor)
                # If decollate_batch returns a 0-d array, wrap in list
                if isinstance(out, list) and len(out) > 0 and getattr(out[0], 'ndim', 1) == 0:
                    return [tensor]
                return out
            except Exception:
                return [tensor]

        preds   = [AsDiscrete(threshold=0.5)(torch.sigmoid(i)) for i in robust_decollate(logits)]
        targets = [AsDiscrete(threshold=0.5)(i)                for i in robust_decollate(labels)]
        self.dice_metric(y_pred=preds, y=targets)

    def on_validation_epoch_end(self):
        dice = self.dice_metric.aggregate().item()
        self.dice_metric.reset()
        self.log("val_dice", dice, prog_bar=True, on_epoch=True, logger=True)

        if self._train_losses_epoch:
            self.train_loss_history.append(float(torch.stack(self._train_losses_epoch).mean()))
            self._train_losses_epoch.clear()
        if self._val_losses_epoch:
            val_loss = float(torch.stack(self._val_losses_epoch).mean())
            self.val_loss_history.append(val_loss)
            self._val_losses_epoch.clear()
        else:
            self.val_loss_history.append(float('nan'))
        self.val_dice_history.append(dice)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr,
            weight_decay=WEIGHT_DECAY, amsgrad=False
        )
        sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            opt, T_0=10, T_mult=1, eta_min=1e-6  # T_0=10 for your 10-epoch test
        )
        return {
            'optimizer': opt,
            'lr_scheduler': {'scheduler': sch, 'interval': 'epoch', 'frequency': 1}
            # NO 'monitor' key — CosineAnnealing doesn't need one
        }

print("LightningModule defined.")


LightningModule defined.


## 6. MONAI Data Transforms
**Fix applied:** `SpatialPadd` pads any volume smaller than `SPATIAL_SIZE` to exactly that size
*before* `RandCropByPosNegLabeld`, so every crop is guaranteed to be `(96, 96, 96)`.
`ResizeWithPadOrCropd` at the end is a final safety net that clamps anything still off-size.
Both transforms use constant (zero) padding so no spurious intensities are introduced.


In [37]:
def get_train_transforms(spatial_size=SPATIAL_SIZE, num_samples=NUM_SAMPLES):
    return Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.5, 1.5, 2.0),
            mode=("bilinear", "nearest"),
        ),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        ScaleIntensityRanged(
            keys=["image"], a_min=-175, a_max=250,
            b_min=0.0, b_max=1.0, clip=True,
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        # ── FIX 1: pad any axis that is shorter than spatial_size ─────
        SpatialPadd(
            keys=["image", "label"],
            spatial_size=spatial_size,
            mode="constant",          # zero-pad (safe for CT HU after scaling)
        ),
        # ── Now every volume is >= spatial_size, crop is always safe ──
        RandCropByPosNegLabeld(
            keys=["image", "label"], label_key="label",
            spatial_size=spatial_size, pos=1, neg=1,
            num_samples=num_samples,  image_key="image",
            image_threshold=0,
            allow_smaller=False,      # strict — guaranteed by SpatialPadd above
        ),
        # ── FIX 2: final clamp in case RandAffine alters the size ─────
        ResizeWithPadOrCropd(keys=["image", "label"], spatial_size=spatial_size),
        RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.5),
        RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
        RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
        RandAffined(
            keys=["image", "label"], mode=("bilinear", "nearest"), prob=0.5,
            rotate_range=(0.1, 0.1, 0.1), scale_range=(0.1, 0.1, 0.1),
            padding_mode="border",
        ),
        # ── FIX 3: re-clamp after RandAffine (affine can shift size) ──
        ResizeWithPadOrCropd(keys=["image", "label"], spatial_size=spatial_size),
        EnsureTyped(keys=["image", "label"]),
    ])


def get_val_transforms():
    """
    Validation: no augmentation; volumes are NOT cropped — sliding-window
    inference handles arbitrary sizes. We still pad to a minimum so the
    UNet encoder path never sees a feature-map smaller than 1 voxel.
    """
    return Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.5, 1.5, 2.0),
            mode=("bilinear", "nearest"),
        ),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        ScaleIntensityRanged(
            keys=["image"], a_min=-175, a_max=250,
            b_min=0.0, b_max=1.0, clip=True,
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        # Pad so sliding-window has at least one full window to work with
        SpatialPadd(keys=["image", "label"], spatial_size=SPATIAL_SIZE, mode="constant"),
        EnsureTyped(keys=["image", "label"]),
    ])

print("Transforms defined (with SpatialPadd + ResizeWithPadOrCropd fixes).")


Transforms defined (with SpatialPadd + ResizeWithPadOrCropd fixes).


## 7. Prepare Dataset and DataLoaders

In [38]:
# ── Collect paired files ──────────────────────────────────────────────────
image_files = sorted([
    f for f in os.listdir(IMAGES_DIR)
    if f.endswith('.nii.gz') and os.path.getsize(os.path.join(IMAGES_DIR, f)) > 0
])
label_files_set = set(
    f for f in os.listdir(LABELS_DIR)
    if f.endswith('.nii.gz') and os.path.getsize(os.path.join(LABELS_DIR, f)) > 0
)
paired_files = [f for f in image_files if f in label_files_set]

if not paired_files:
    raise ValueError(
        f"No paired .nii.gz files found.\n"
        f"  IMAGES_DIR: {IMAGES_DIR}\n"
        f"  LABELS_DIR: {LABELS_DIR}\n"
        "Check that image and label filenames match exactly."
    )

# ── Train / Val split ─────────────────────────────────────────────────────
if len(paired_files) > 1:
    train_files, val_files = train_test_split(
        paired_files, test_size=VAL_SPLIT, random_state=RANDOM_SEED
    )
else:
    train_files = val_files = paired_files   # single-volume fallback

train_data = [{"image": os.path.join(IMAGES_DIR, f), "label": os.path.join(LABELS_DIR, f)} for f in train_files]
val_data   = [{"image": os.path.join(IMAGES_DIR, f), "label": os.path.join(LABELS_DIR, f)} for f in val_files]

print(f"Total paired files : {len(paired_files)}")
print(f"Training samples   : {len(train_data)}")
print(f"Validation samples : {len(val_data)}")

# ── Datasets & DataLoaders ────────────────────────────────────────────────
train_ds = Dataset(data=train_data, transform=get_train_transforms())
val_ds   = Dataset(data=val_data,   transform=get_val_transforms())

# NOTE: BATCH_SIZE=1 is the safe default for 3D volumes.
# All training crops are now guaranteed (96,96,96) so batch stacking works.
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=8,        # AMD CPU with plenty of cores — use 8
    pin_memory=True,      # fast PCIe transfer to RTX 5090
    persistent_workers=True,  # avoid respawning workers each epoch
    prefetch_factor=2,
)
val_loader = DataLoader(
    val_ds, batch_size=1, shuffle=False,
    num_workers=0, pin_memory=False,
)

print("DataLoaders ready.")


Total paired files : 421
Training samples   : 336
Validation samples : 85
DataLoaders ready.


## 8. Train the Model

In [39]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_dice",
    mode="max",
    save_top_k=1,          # keep only the single best checkpoint
    dirpath=OUTPUT_DIR,
    filename="unet3d-best-{epoch:02d}-{val_dice:.4f}",
    verbose=True,
    save_last=True,        # always also save the latest epoch
)
early_stop_callback = EarlyStopping(
    monitor="val_dice",
    patience=10,
    mode="max",
    verbose=True,
)
logger = TensorBoardLogger(save_dir=OUTPUT_DIR, name="unet3d_training")

trainer = pl.Trainer(
    accelerator='gpu',        # force GPU explicitly
    devices=1,
    precision='bf16-mixed',   # RTX 5090 has native BF16 — more stable than fp16
    gradient_clip_val=1.0,
    accumulate_grad_batches=ACCUM_GRAD,
    max_epochs=MAX_EPOCHS,
    log_every_n_steps=1,
    check_val_every_n_epoch=2,  # validate every 2 epochs to save time
    enable_progress_bar=True,
    logger=logger,
    callbacks=[checkpoint_callback, early_stop_callback],
)

model = TumorSegmentation3D(lr=1e-4)
print("Starting training...")
trainer.fit(model, train_loader, val_loader)
print(f"\nTraining complete!")
print(f"Best checkpoint  : {checkpoint_callback.best_model_path}")
print(f"Best val Dice    : {checkpoint_callback.best_model_score:.4f}")


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Starting training...


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ UNet3D     │ 23.3 M │ train │     0 │
│ 1 │ loss_fn │ DiceCELoss │      0 │ train │     0 │
└───┴─────────┴────────────┴────────┴───────┴───────┘

Trainable params: 23.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.3 M                                                                                               
Total estimated model params size (MB): 93                                                                         
Modules in train mode: 63                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

## 9. Training Curves — Loss & Dice

In [ ]:
epochs = list(range(1, len(model.train_loss_history) + 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("3D UNet Training — NSCLC Radiomics", fontsize=14, fontweight="bold")

axes[0].plot(epochs, model.train_loss_history, marker='o', color='steelblue', linewidth=2)
axes[0].set_title("Training Loss");  axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("DiceCE Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, model.val_loss_history, marker='s', color='tomato', linewidth=2)
axes[1].set_title("Validation Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("DiceCE Loss")
axes[1].grid(True, alpha=0.3)

best_epoch = int(np.argmax(model.val_dice_history)) + 1
best_dice  = max(model.val_dice_history)
axes[2].plot(epochs, model.val_dice_history, marker='^', color='mediumseagreen', linewidth=2, label='Val Dice')
axes[2].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch={best_epoch}')
axes[2].scatter([best_epoch], [best_dice], color='red', zorder=5, s=80, label=f'Best={best_dice:.4f}')
axes[2].set_title("Validation Dice Score"); axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Dice")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "training_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Curves saved → {plot_path}")

print(f"\n{'Epoch':>6} | {'Train Loss':>12} | {'Val Loss':>10} | {'Val Dice':>10}")
print("-" * 50)
for i, (tl, vl, vd) in enumerate(zip(model.train_loss_history, model.val_loss_history, model.val_dice_history), 1):
    tag = " ◀ best" if i == best_epoch else ""
    print(f"{i:>6} | {tl:>12.4f} | {vl:>10.4f} | {vd:>10.4f}{tag}")


## 10. Load Best Model for Inference

In [ ]:
best_ckpt_path = checkpoint_callback.best_model_path
print(f"Loading best checkpoint: {best_ckpt_path}")

best_model = TumorSegmentation3D.load_from_checkpoint(best_ckpt_path)
best_model.eval()
best_model.to(device)
print("Best model loaded and ready for inference.")